In [1]:
import json
import re
from pathlib import Path

import pandas as pd
from datasets import load_dataset

In [2]:
RUNS_DIR = Path("runs")

# question_type is looked up from the dataset, so older eval files without it still work
qa = load_dataset("Liadmagen/financebench_QA")["train"]
question_types = {r["financebench_id"]: r["question_type"] for r in qa}

REFUSAL = re.compile(r"cannot be answered|insufficient|not (?:be )?(?:answered|available|provided)|unable", re.I)

Using the latest cached version of the dataset since Liadmagen/financebench_QA couldn't be found on the Hugging Face Hub (offline mode is enabled).


Found the latest cached dataset configuration 'default' at /Users/rebeccadifrancesco/.cache/huggingface/datasets/Liadmagen___financebench_qa/default/0.0.0/5645a7fd9ddd9d5353566d1e41e472df50c9ef05 (last modified on Sun Nov  1 21:34:08 2026).


In [3]:
def load_all_results():
    rows = []

    for f in sorted(RUNS_DIR.glob("*/eval.json")):
        try:
            with f.open() as fh:
                for r in json.load(fh):
                    if isinstance(r, dict) and "correct" in r:
                        rows.append({
                            "run": f.parent.name,
                            "financebench_id": r["financebench_id"],
                            "question_type": question_types.get(r["financebench_id"], "unknown"),
                            "correct": r["correct"],
                            "refused": bool(REFUSAL.search(r["llm_answer"])),
                            "error": r["llm_answer"].startswith("ERROR"),
                        })
        except Exception as e:
            print(f"⚠️ Skipping {f}: {e}")

    return pd.DataFrame(rows)

df = load_all_results()

In [4]:
if df.empty:
    print("No judged results found.")
else:
    summary = df.groupby("run").agg(
        evaluated=("correct", "size"),
        correct=("correct", "sum"),
        accuracy=("correct", "mean"),
        refusals=("refused", "sum"),
        errors=("error", "sum"),
    )
    print("FINAL ACCURACY REPORT")
    print("=" * 30)
    display(summary.style.format({"accuracy": "{:.2%}"}))

FINAL ACCURACY REPORT


,evaluated,correct,accuracy,refusals,errors
run,,,,,
full,150,128,85.33%,0,0
quick2,30,25,83.33%,0,0


In [5]:
# Accuracy per FinanceBench question type
by_type = df.pivot_table(index="question_type", columns="run", values="correct", aggfunc="mean")
display(by_type.style.format("{:.2%}"))

run,full,quick2
question_type,,
domain-relevant,86.00%,70.00%
metrics-generated,90.00%,87.50%
novel-generated,80.00%,91.67%


In [6]:
# Compare runs on the same questions only (e.g. when a run used --limit)
runs = df["run"].unique()
if len(runs) > 1:
    common = set.intersection(*(set(df.loc[df["run"] == r, "financebench_id"]) for r in runs))
    same = df[df["financebench_id"].isin(common)]
    print(f"Questions judged in every run: {len(common)}")
    display(same.groupby("run")["correct"].mean().to_frame("accuracy").style.format("{:.2%}"))

Questions judged in every run: 30


,accuracy
run,
full,86.67%
quick2,83.33%
